# Blood Pressure Modeling (For Inferential Statistics)

## Prepare Data

In [52]:
import pandas as pd
df = pd.read_csv('cleaned_blood_pressure_data.csv')

In [56]:
df.columns

Index(['participant_id', 'bp_cuff_size', 'weight_kg', 'height_cm', 'bmi',
       'age', 'gender', 'race_ethnicity', 'education_level',
       'income_poverty_ratio', 'marital_status', 'mec_exam_weight',
       'survey_stratum', 'survey_psu', 'sbp_mean', 'dbp_mean', 'sodium_mean',
       'potassium_mean', 'magnesium_mean', 'calcium_mean', 'energy_mean',
       'alcohol_mean', 'caffeine_mean', 'map'],
      dtype='object')

In [57]:
# Encode binary variable
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})

# One-hot encode race_ethnicity & marital_status
df = pd.get_dummies(df, columns=['race_ethnicity','marital_status'], drop_first=True)

# Drop IDs or other irrelevant columns
df.drop(columns=['participant_id', 'bp_cuff_size', 'weight_kg', 'height_cm', 
                 'mec_exam_weight', 'survey_stratum', 'survey_psu'], inplace=True)

# Define target and features
X = df.drop(columns=['sbp_mean', 'dbp_mean', 'map'])  # Keep only predictors
y = df['dbp_mean'] 

In [58]:
df.head()

,bmi,age,gender,education_level,income_poverty_ratio,sbp_mean,dbp_mean,sodium_mean,potassium_mean,magnesium_mean,...,alcohol_mean,caffeine_mean,map,race_ethnicity_Non-Hispanic Black,race_ethnicity_Non-Hispanic White,race_ethnicity_Other Hispanic,race_ethnicity_Other Race - Including Multi-Racial,marital_status_Missing,marital_status_Never married/Single,marital_status_Widowed/Divorced/Separated
0,27.0,43.0,0,5.0,5.00,132.666667,96.000000,2144.0,1889.0,208.0,...,2.280000e+01,160.5,108.222222,0,0,0,1,0,0,0
1,33.5,66.0,0,5.0,5.00,117.000000,78.666667,4536.5,4515.0,508.5,...,6.030000e+01,7.0,91.444444,0,1,0,0,0,0,0
2,29.7,44.0,1,3.0,1.41,109.000000,78.333333,2934.5,2053.5,353.0,...,5.397605e-79,77.0,88.555556,0,0,1,0,0,0,0
3,30.2,34.0,0,4.0,1.33,115.000000,73.666667,3520.5,1991.0,209.0,...,9.300000e+00,240.0,87.444444,0,0,0,0,0,0,0
4,42.6,68.0,1,5.0,1.32,141.333333,76.000000,7097.5,4593.0,315.5,...,5.397605e-79,93.5,97.777778,0,1,0,0,0,1,0


## Ridge Regression

In [59]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Split into train/test (optional)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge regression with cross-validation to choose alpha
alphas = [0.1, 1, 10, 50, 100]  # you can expand this list
ridge_cv = RidgeCV(alphas=alphas, scoring='r2', store_cv_values=True)

# Scaling + Ridge pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', ridge_cv)
])

# Fit model
pipeline.fit(X_train, y_train)

# Best alpha
print("Best alpha:", pipeline.named_steps['ridge'].alpha_)

# Coefficients
coeffs = pd.Series(pipeline.named_steps['ridge'].coef_, index=X_train.columns)
print(coeffs.sort_values(ascending=False))

# Evaluate R^2 on test set
r2 = pipeline.score(X_test, y_test)
print("Test R^2:", r2)


Best alpha: 50.0
bmi                                                   2.476316
race_ethnicity_Non-Hispanic Black                     1.238433
race_ethnicity_Other Race - Including Multi-Racial    0.782108
alcohol_mean                                          0.681154
race_ethnicity_Non-Hispanic White                     0.432832
race_ethnicity_Other Hispanic                         0.294291
energy_mean                                           0.259396
income_poverty_ratio                                  0.205978
caffeine_mean                                         0.184521
marital_status_Widowed/Divorced/Separated             0.029271
marital_status_Never married/Single                  -0.021810
sodium_mean                                          -0.117464
calcium_mean                                         -0.139749
age                                                  -0.174068
potassium_mean                                       -0.178552
magnesium_mean                        

## Assumptions for Regression

### Multicollinearity

In [38]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif['feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

                                              feature          VIF
0                                        bp_cuff_size    99.315624
1                                           weight_kg   406.946771
2                                           height_cm  1106.991780
3                                                 bmi   440.281549
4                                                 age    17.215273
5                                              gender     3.554680
6                                     education_level    22.958828
7                                income_poverty_ratio     7.004125
8                                     mec_exam_weight     3.925144
9                                      survey_stratum  1039.315538
10                                         survey_psu    10.219088
11                                        sodium_mean    24.277646
12                                     potassium_mean    28.992586
13                                     magnesium_mean    24.28

In [42]:
X.drop(columns=['bp_cuff_size','weight_kg', 'height_cm', 'survey_stratum', 'survey_psu', 'mec_exam_weight'], inplace=True)

In [45]:
X.drop(columns=['education_level','energy_mean'], inplace=True)

In [47]:
X.drop(columns=['potassium_mean'], inplace=True)

In [49]:
X.drop(columns=['magnesium_mean'], inplace=True)

In [50]:
vif = pd.DataFrame()
vif['feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

                                              feature        VIF
0                                                 bmi  12.327476
1                                                 age  10.135225
2                                              gender   2.278025
3                                income_poverty_ratio   4.557905
4                                         sodium_mean  10.746181
5                                        calcium_mean   7.957140
6                                        alcohol_mean   1.222696
7                                       caffeine_mean   2.164686
8                   race_ethnicity_Non-Hispanic Black   2.271539
9                   race_ethnicity_Non-Hispanic White   7.206458
10                      race_ethnicity_Other Hispanic   2.018470
11  race_ethnicity_Other Race - Including Multi-Ra...   2.257171
12                             marital_status_Missing   2.320979
13                marital_status_Never married/Single   1.469282
14          marital_statu

## Simple Linear Modeling

In [51]:
import statsmodels.api as sm

# Add constant for intercept
X_sm = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X_sm).fit()

# View results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               dbp_mean   R-squared:                       0.233
Model:                            OLS   Adj. R-squared:                  0.231
Method:                 Least Squares   F-statistic:                     150.3
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        18:58:23   Log-Likelihood:                -27753.
No. Observations:                7453   AIC:                         5.554e+04
Df Residuals:                    7437   BIC:                         5.565e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------